# Context Management for the MiniClaw
## ME 493B — AI in Product Development | Mini-Project 2, Part B

**Instructor:** Scott Thielman, PhD — University of Washington Bothell
**Due:** Monday, April 27, 2026 at 11:59 PM
**Points:** 50 (Part B). Part A is worth 50 points separately.

---

### What this notebook is

This is the research phase of the MiniClaw project. Jordan Chen has reviewed
your MP1 gear train designs and wants deeper research before the team moves
to detailed design. Read the memo from Jordan (`MP2_PartB_Research_Directive.docx`)
before starting.

You have access to ACME's internal knowledge base — 15 documents covering
manufacturing capabilities, material test data, previous product history,
engineering standards, and vendor information. Your job is to demonstrate
that **managing context** produces better AI-assisted engineering answers
than working from general knowledge alone.

### How to use this notebook

This notebook provides the structure and the data. **You fill in every
section.** Use markdown cells for written deliverables and code cells for
your RAG pipeline work. Use GitHub Copilot, Claude, ChatGPT, or any AI
tool — the learning is in directing the AI with the right context and
evaluating whether the output is trustworthy.

The RAG pipeline you built in Part A is one approach for managing the ACME
context. You may reuse that pipeline, adapt it, or use any approach you choose.

### Grading summary (50 pts)

| Deliverable | Points |
|-------------|--------|
| 1. Project Intake Document | 10 |
| 2. Information Strategy | 8 |
| 3. Context Package + Before/After Demo | 15 |
| 4. Research Synthesis | 7 |
| 5. Working PRD | 10 |
| **Total** | **50** |

### How to submit

1. Complete all sections in this notebook
2. Make sure all cells run top-to-bottom without errors
3. **Commit and push** to your course repository via the Source Control panel
4. Verify your push succeeded on GitHub (check that your latest changes appear)
5. Submit your GitHub repo URL on Canvas (same URL as Part A)

---
## Section 0: Setup

Run **both** setup cells below before starting. Cell 2 loads the ACME corpus;
Cell 3 installs packages, loads your GitHub token from `.env`, and initializes
the embedding model and API client. Do not modify these cells.

> **GitHub token required:** The `call_github_model` helper uses the
> GitHub Models API (free tier). You need a fine-grained Personal Access
> Token with **Account permissions → Models → Read-only**. Create one at
> [github.com/settings/tokens](https://github.com/settings/tokens) and
> add it to the `.env` file in your repo root:
> ```
> GITHUB_TOKEN=ghp_your_token_here
> ```


In [1]:
# ── Setup: load the ACME corpus and standard libraries ──────────────────
# Run this cell first. Do not modify.

import sys, os, json

# If running from the MP2/Part B folder, the corpus is right here.
# If running from the repo root, adjust the path.
_nb_dir = os.path.abspath("")
if os.path.exists(os.path.join(_nb_dir, "corpus", "manifest.json")):
    _corpus_dir = _nb_dir
elif os.path.exists(os.path.join(_nb_dir, "MP2", "Part B", "corpus", "manifest.json")):
    _corpus_dir = os.path.join(_nb_dir, "MP2", "Part B")
else:
    raise FileNotFoundError(
        "Cannot find corpus/ folder. Make sure you are running this notebook "
        "from the MP2/Part B/ directory or the repository root."
    )

sys.path.insert(0, _corpus_dir)
from acme_miniclaw_corpus import acme_documents

print(f"Loaded {len(acme_documents)} ACME documents")
print(f"Total characters: {sum(len(d['text']) for d in acme_documents):,}")
print()
for doc in acme_documents:
    print(f"  {doc['id']:15s} {doc['title'][:60]:60s} ({len(doc['text']):,} chars)")

Loaded 15 ACME documents
Total characters: 37,396

  ACME-MFG-001    Print Shop Capabilities & Standards                          (2,282 chars)
  ACME-MFG-002    FilaTech PolyPro PLA+ Technical Summary                      (2,219 chars)
  ACME-MFG-003    3D Printed Gear Test Report — WidgetBot 2.0 Project          (3,045 chars)
  ACME-MFG-004    Assembly Line Guidelines for 3D Printed Products             (2,540 chars)
  ACME-PROD-001   WidgetBot 2.0 Design Summary                                 (2,501 chars)
  ACME-PROD-002   GripperBot Product Specification                             (2,458 chars)
  ACME-PROD-003   Component Library Catalog — Q1 2026                          (2,137 chars)
  ACME-BIZ-001    RobotExpo 2026 Marketing Brief                               (2,274 chars)
  ACME-BIZ-002    RobotExpo 2026 Logistics Memo                                (2,040 chars)
  ACME-BIZ-003    Q1 2026 Engineering Status Report                            (2,321 chars)
  ACME-ENG-001    G

In [2]:
# ── Add any additional imports or setup here ────────────────────────────
# Install packages needed for the RAG pipeline (if not already installed)
import subprocess, sys
for pkg in ["sentence-transformers", "python-dotenv"]:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

# Load GITHUB_TOKEN from .env (repo root) — needed for LLM generation calls.
# Add your token to .env: GITHUB_TOKEN=ghp_...
# Fine-grained token: Account permissions → Models → Read-only
import os, pathlib
from dotenv import load_dotenv
for candidate in [pathlib.Path(".env"), pathlib.Path("../../.env")]:
    if candidate.exists():
        load_dotenv(candidate)
        break

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
if not GITHUB_TOKEN:
    print("WARNING: GITHUB_TOKEN not set — LLM calls will fail.")
    print("Set it in .env at the repo root: GITHUB_TOKEN=ghp_...")
else:
    print(f"GitHub token loaded ({len(GITHUB_TOKEN)} chars)")

# Common imports for the RAG pipeline
import chromadb
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
import textwrap, time

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

github_client = OpenAI(
    base_url="https://models.github.ai/inference",
    api_key=GITHUB_TOKEN,
)

def call_github_model(messages, model="openai/gpt-4o", max_retries=3):
    """Call GitHub Models API with retry on rate limits."""
    for attempt in range(max_retries):
        try:
            response = github_client.chat.completions.create(
                model=model, messages=messages, temperature=0.3, max_tokens=600,
            )
            return response.choices[0].message.content
        except Exception as e:
            if "rate" in str(e).lower() and attempt < max_retries - 1:
                wait = 15 * (attempt + 1)
                print(f"  Rate limited — waiting {wait}s...")
                time.sleep(wait)
            else:
                raise

print("Setup complete. Embedding model loaded.")


GitHub token loaded (40 chars)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setup complete. Embedding model loaded.


---
## Deliverable 1: Project Intake Document (10 pts)

Apply the **PCS intake framework** (from Session 5) to the MiniClaw project.
This is the document that would kick off the project in a real engineering firm.

**Include:**
- **Vision** — what does ACME want the MiniClaw to achieve? (Use the ACME corpus
  for business context, not just the original design brief.)
- **Mission** — your specific engineering scope
- **In scope / Out of scope** — what are you designing vs. what is someone else's
  problem?
- **Key assumptions** — what are you taking as given?
- **Responsibilities** — who owns what?
- **Phase Zero recommendation** — should the project proceed directly to design,
  or does it need more exploration first? Justify your answer.

**Note:** The MiniClaw design brief from MP1 is your starting point, but your
intake should reflect the deeper context from the ACME corpus — company
capabilities, manufacturing constraints, business goals for RobotExpo.

*Write your intake document in the markdown cell below. Use headings to
organize the sections.*

### Project Intake: MiniClaw

#### Vision
ACME Robotics wants the MiniClaw to function as a high-quality marketing asset and engineering "business card" at RobotExpo 2026 (June 10–12). Per Sarah Kim's marketing brief (ACME-BIZ-001), it must convince technically literate attendees — engineering students, makers, robotics educators — that "these people really know gears and mechanisms." The MiniClaw is not just a toy; per Sarah, "a gripper that jams or feels cheap does more harm than not having a giveaway at all." The MiniClaw should also reflect ACME's actual production realities: 3D printed in PLA on the in-house Prusa MK4S fleet, manufactured economically at quantity 500 units within a $3.50/unit material budget.

#### Mission
Design a hand-driven, gear-actuated PLA gripper (the MiniClaw) that:
- Demonstrates ACME's mechanical engineering credibility through a smooth, satisfying, mechanically meaningful gear+linkage feel
- Is producible on ACME's existing Prusa MK4S fleet using FilaTech PolyPro PLA+ within Maria Santos's documented tolerances
- Hits the 500-unit RobotExpo production schedule (final design files to fab team by May 15, 2026)
- Stays within the $3.50/unit material cost target and uses the existing component library where helpful

#### In Scope
- Gear train design (module, tooth counts, ratios, face widths, derating per ACME-ENG-001)
- Linkage design (4-bar) converting gear output to jaw motion
- Housing / chassis to package the mechanism, including ACME logo emboss
- Tolerance / press-fit / clearance scheme for 3D-printed assembly
- Material selection (within ACME's stocked filaments)
- Bill of materials within $3.50/unit target
- Use of in-house component library items (brass pins, silicone grip pads, etc.) — pending Jordan's confirmation per ACME-PROD-003
- Test plan: clamping force at 25 mm jaw opening (per GripperBot SOP), gear durability, jaw alignment

#### Out of Scope
- Servo or electronics integration (MiniClaw is hand-driven; this is a key difference from GripperBot/BigClaw)
- Injection-molded or CNC-machined parts (production is FDM only)
- Custom filament development or new material qualification (use stocked PolyPro PLA+)
- Marketing collateral beyond the embossed logo
- End-user assembly instructions/packaging (Marketing owns the header card content)
- Post-RobotExpo product line extension or commercial pricing

#### Key Assumptions
- ACME PolyPro PLA+ properties hold (interlayer adhesion 28 ±2 MPa governs gear teeth, per ACME-MFG-002)
- Prusa MK4S, 0.4 mm nozzle, 0.15–0.20 mm layer heights, 100% infill on gears (ACME-MFG-001 / ACME-ENG-001)
- Component-library items (brass pins, silicone grip pads) are usable pending Jordan's confirmation re: "no purchased parts except input wheel" rule
- ~35 g of PLA per unit, ~$0.76/unit filament cost based on FilaTech estimate (ACME-VND-001) — leaves room in the $3.50 BOM
- PolyPro PLA+ Company Blue requires a 3-week PO lead time, so the filament order must go out by ~mid-April to be in hand by early May
- Jaw opening target ~50–60 mm and clamping force on the order of 5–8 N at mid-travel (scaled from BigClaw 86 mm / GripperBot 8 N benchmarks)

#### Responsibilities
- **Engineer (me):** Gear train + linkage + housing design, tolerance analysis, BOM, design review prep
- **Maria Santos (Fab Team Lead):** Print shop scheduling, tolerance feasibility sign-off, 500-unit production
- **Ryan Cooper (Materials Engineering):** PolyPro PLA+ data, FilaTech ordering
- **Dr. Elena Vasquez (Test Engineering):** Gear fatigue / durability protocol, test fixtures (referenced from WidgetBot 2.0)
- **Kevin Park (Lead ME):** Internal standard ACME-ENG-001 owner, component-library guidance, BigClaw teardown reference
- **Jordan Chen (Engineering Manager):** Concept-to-Detailed-Design review, "no purchased parts" clarification
- **Sarah Kim (Marketing):** Branding/colors/logo requirements, packaging
- **Tom Bradley (Operations):** RobotExpo logistics
- **Jamie Wu (FilaTech, external):** Custom Blue PLA color matching + bulk delivery

#### Phase Zero Recommendation
**Recommendation: Brief, focused exploration before detailed design — do NOT proceed straight to detailed design yet.** Jordan's memo explicitly says MP1 designs were built on generic data and need rework; the research phase exists precisely to fix that. However, "exploration" does not mean redesign from scratch — it means a one-week deep read of ACME-ENG-001, ACME-MFG-002/003, and ACME-PROD-001/002/003 to ground assumptions, plus an open-question to Jordan on the component-library carve-out. With that done, the project has a solid path to detailed design and can still hit the May 15 fab handoff. The only schedule risk is the 3-week Company Blue PLA lead time, which means the filament PO must be triggered in parallel with concept design, not after.

---
## Deliverable 2: Information Strategy (8 pts)

Answer: **what information do you need to design the MiniClaw well?**

Categorize into three buckets:

| Bucket | Description | Examples needed |
|--------|-------------|-----------------|
| **AI knows well** | General engineering knowledge the AI can be trusted on | 2–3 MiniClaw-specific examples |
| **AI knows partially** | Correct in general, may be wrong in specifics | 2–3 MiniClaw-specific examples |
| **Requires ACME context** | Only available in the internal knowledge base | 2–3 MiniClaw-specific examples |

This demonstrates you understand **WHY** context management matters,
not just how to do it.

*Write your information strategy in the markdown cell below.*

### Information Strategy

Designing the MiniClaw well means routing each question to a source the AI is actually trustworthy on. Below I bucket the information needs into three categories. The deeper point is that **AI confidence does not vary by category — only correctness does.** A general-purpose LLM will speak with equal authority about generic Lewis bending stress and about ACME's interlayer adhesion number. Without context management, I can't tell those answers apart.

| Bucket | Description | MiniClaw-specific examples |
|--------|-------------|----------------------------|
| **AI knows well** | Settled, textbook engineering: standard formulas, conventional design heuristics, and well-established mechanism kinematics that don't depend on a specific shop or material. The AI is reliable here. | (a) The Lewis bending stress equation and how to apply it to a spur gear tooth. (b) The kinematics of a 4-bar linkage given crank/coupler/output/ground link lengths, including how to compute mechanical advantage through travel. (c) The relationship between module, pitch diameter, and tooth count for a spur gear set (a = m(z1+z2)/2). |
| **AI knows partially** | Correct in the general direction but wrong on specifics that depend on material lot, process, or shop-specific empirical behavior. The AI will give plausible numbers that don't apply to *our* parts. | (a) "Safety factor for plastic gears" — generic answer is ~1.5; ACME data (ACME-MFG-003) shows we need 2.0 for printed PLA gears because the failure mode is layer delamination, not tooth root fracture. (b) "PLA tensile strength" — datasheets vary 40–70 MPa; our PolyPro PLA+ is 52 MPa bulk but only **28 MPa interlayer**, and 28 MPa is the governing number for printed gear teeth. (c) "Minimum tooth count for a spur gear at 20° pressure angle" — textbook says 17 for no undercutting; ACME-ENG-001 allows 14 at module 1.0 in PLA based on what actually prints reliably on our MK4S fleet. |
| **Requires ACME context** | Information that exists only inside ACME's knowledge base and could not be guessed by any general-purpose model. Skipping these is what produces the rework Jordan called out. | (a) Achievable press-fit tolerance on the MK4S fleet (±0.15 mm per ACME-MFG-001) — directly drives nominal pin/bore sizing for brass pins from the component library. (b) WidgetBot 2.0 lessons — minimum face width 4 mm, contact ratio >1.5 for quiet operation, 0.10–0.15 mm backlash to survive thermal cycling, 100% infill non-negotiable (ACME-PROD-001). (c) The $3.50/unit material budget, RobotExpo 500-unit volume, and FilaTech 3-week lead time on Company Blue PLA — sets non-negotiable constraints on color choice and order timing (ACME-BIZ-001, ACME-VND-001). (d) Existence and dimensions of in-house brass pin stock and silicone grip pads in the component library — and the open question about the "no purchased parts" rule (ACME-PROD-003). |

**Why this matters:** The questions in the third bucket are exactly the ones where MP1 designs went wrong, and they are exactly the questions where a no-context AI will hallucinate a confident-sounding answer. Context management — chunking the ACME corpus into a RAG pipeline and routing those questions to it — is the engineering control that prevents that failure mode.

---
## Deliverable 3: Context Package + Before/After Demo (15 pts)

This is the **core deliverable**. Two steps:

### Step 1: Build your context package

Load the ACME document corpus into a RAG pipeline. You can reuse or adapt
your Part A pipeline, or build something new. Show your work in the code
cells below.

> **Reminder from Part A:** ChromaDB defaults to L2 (Euclidean) distance.
> Since your embeddings use cosine similarity, create your collection with
> `metadata={"hnsw:space": "cosine"}` to get correct retrieval rankings.

### Step 2: Before/After demonstration

Choose **3 specific MiniClaw engineering questions** where ACME context
would matter. For each question, show:

1. The AI's answer **WITHOUT** the ACME context — call `call_github_model`
   directly with the question and no retrieved chunks. This shows what the
   AI knows from general training alone.
2. The AI's answer **WITH** the ACME context — use your RAG pipeline to
   retrieve relevant chunks, then pass them as context to `call_github_model`.
3. **Your assessment:** What changed? Is the augmented answer better? What
   did it get right that the baseline missed?

**Choose meaningful engineering questions, not trivia.**
- ✅ Good: "What safety factor should I use for 3D-printed PLA gears?"
- ✅ Good: "What PLA printing parameters should I use for the MiniClaw gears?"
- ❌ Bad: "What is ACME's address?"


### Step 1: Build context package

Use the code cell below to:
1. Define a `chunk_text(text, chunk_size, overlap)` function (or import your Part A version)
2. Chunk all 15 ACME documents
3. Create a ChromaDB collection with `metadata={"hnsw:space": "cosine"}`
4. Embed all chunks with `embedding_model` and add them to the collection
5. Write a `query_rag(question, n_results=3)` function

You may copy and adapt your Part A solution, or rebuild from scratch.


In [3]:
# ── Build the RAG pipeline / context package over the ACME corpus ──────
# Reusing my Part A approach: 500-char chunks with 100-char overlap, ChromaDB
# with cosine similarity, all-MiniLM-L6-v2 embeddings.

CHUNK_SIZE = 500
OVERLAP = 100

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP, doc_id=None, doc_title=None):
    """Split text into overlapping chunks; preserve source metadata."""
    chunks = []
    start = 0
    chunk_index = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append({
            "text": chunk,
            "doc_id": doc_id,
            "doc_title": doc_title,
            "chunk_index": chunk_index,
        })
        start += chunk_size - overlap
        chunk_index += 1
    return chunks

# Chunk all 15 ACME documents
all_chunks = []
for doc in acme_documents:
    all_chunks.extend(chunk_text(doc["text"], doc_id=doc["id"], doc_title=doc["title"]))

print(f"Created {len(all_chunks)} chunks from {len(acme_documents)} documents")

# Build ChromaDB collection (cosine space — Part A reminder)
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("acme_miniclaw")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="acme_miniclaw",
    metadata={"hnsw:space": "cosine"},
)

# Embed and add
chunk_texts = [c["text"] for c in all_chunks]
chunk_embeddings = embedding_model.encode(chunk_texts).tolist()
collection.add(
    ids=[f"{c['doc_id']}_chunk{c['chunk_index']}" for c in all_chunks],
    embeddings=chunk_embeddings,
    documents=chunk_texts,
    metadatas=[{"doc_id": c["doc_id"], "doc_title": c["doc_title"], "chunk_index": c["chunk_index"]} for c in all_chunks],
)
print(f"ChromaDB collection 'acme_miniclaw' indexed: {collection.count()} chunks")

def query_rag(question, n_results=3):
    """Embed question, return top-N chunks with metadata."""
    q_emb = embedding_model.encode([question]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=n_results)
    retrieved = []
    for i in range(len(results["documents"][0])):
        retrieved.append({
            "text": results["documents"][0][i],
            "doc_id": results["metadatas"][0][i]["doc_id"],
            "doc_title": results["metadatas"][0][i]["doc_title"],
            "chunk_index": results["metadatas"][0][i]["chunk_index"],
            "distance": results["distances"][0][i] if results.get("distances") else None,
        })
    return retrieved

def answer_with_context(question, n_results=3):
    """Retrieve ACME chunks, then ask the LLM to answer grounded in them with citations."""
    chunks = query_rag(question, n_results=n_results)
    context = "\n\n".join(
        f"[Chunk {i+1} from {c['doc_id']} — {c['doc_title']}]:\n{c['text']}"
        for i, c in enumerate(chunks)
    )
    messages = [
        {"role": "system", "content": (
            "You are a mechanical engineering assistant working at ACME Robotics on the MiniClaw project. "
            "Answer ONLY based on the provided ACME context. Cite the document IDs you used (e.g., ACME-ENG-001). "
            "If the context does not contain the answer, say 'The provided ACME context does not contain this information.'"
        )},
        {"role": "user", "content": f"ACME context:\n{context}\n\nQuestion: {question}"},
    ]
    return call_github_model(messages), chunks

def answer_without_context(question):
    """Ask the LLM the same question with no ACME context — baseline general knowledge."""
    messages = [
        {"role": "system", "content": (
            "You are a mechanical engineering assistant. Answer the question using your general engineering knowledge."
        )},
        {"role": "user", "content": question},
    ]
    return call_github_model(messages)

# Smoke-test
test = query_rag("PLA gear safety factor")
print(f"\nSmoke-test query returned {len(test)} chunks. Top hit: [{test[0]['doc_id']}] {test[0]['text'][:80]}...")

Created 102 chunks from 15 documents


ChromaDB collection 'acme_miniclaw' indexed: 102 chunks

Smoke-test query returned 3 chunks. Top hit: [ACME-MFG-003] Layer height: 0.20 mm
- Print orientation: gear axis perpendicular to build plat...


### Step 2: Before/After Demo

#### Question 1

In [4]:
# ── Question 1: WITHOUT ACME context ────────────────────────────────────
# Q1: What safety factor and allowable stress should I use for 3D-printed
#     PLA spur gears in the MiniClaw, and why?
question_1 = ("What safety factor and allowable stress should I use for 3D-printed PLA spur gears "
              "in a hand-driven gripper mechanism, and what is the dominant failure mode?")

print("=" * 75)
print("QUESTION 1 (no ACME context):")
print(question_1)
print("=" * 75)
answer_1_baseline = answer_without_context(question_1)
print(answer_1_baseline)

QUESTION 1 (no ACME context):
What safety factor and allowable stress should I use for 3D-printed PLA spur gears in a hand-driven gripper mechanism, and what is the dominant failure mode?


When designing 3D-printed PLA spur gears for a hand-driven gripper mechanism, the safety factor, allowable stress, and failure mode depend on several factors, including the loading conditions, operating environment, and material properties. Here's a detailed breakdown:

### 1. **Safety Factor (SF):**
   - For 3D-printed components, a **safety factor of 2 to 3** is commonly recommended due to the variability in material properties, anisotropy caused by the printing process, and potential defects such as voids or layer adhesion issues.
   - Since the mechanism is hand-driven and likely operates under relatively low loads, a safety factor closer to **2** may be sufficient if the load is well-characterized and the gears are not subjected to impact or fatigue. However, if there is any uncertainty in the loading or environmental conditions, a higher safety factor (closer to 3) is advisable.

### 2. **Allowable Stress:**
   - The allowable stress for PLA depends on its tensile strength and th

In [5]:
# ── Question 1: WITH ACME context ───────────────────────────────────────
print("=" * 75)
print("QUESTION 1 (with ACME context):")
print(question_1)
print("=" * 75)
answer_1_rag, retrieved_1 = answer_with_context(question_1, n_results=4)
print("Retrieved chunks:")
for i, c in enumerate(retrieved_1, 1):
    print(f"  {i}. [{c['doc_id']}] {c['text'][:110].strip()}...")
print()
print("Answer:")
print(answer_1_rag)

QUESTION 1 (with ACME context):
What safety factor and allowable stress should I use for 3D-printed PLA spur gears in a hand-driven gripper mechanism, and what is the dominant failure mode?


Retrieved chunks:
  1. [ACME-MFG-003] Layer height: 0.20 mm
- Print orientation: gear axis perpendicular to build plate (teeth oriented perpendicula...
  2. [ACME-MFG-003] eth failed at approximately 60% of the stress predicted by classical Lewis theory using bulk PLA properties. T...
  3. [ACME-MFG-002] he governing failure mode for any printed part where loads act perpendicular to the layer lines — including ge...
  4. [ACME-MFG-003] TEST REPORT: 3D PRINTED SPUR GEAR DURABILITY
Project: WidgetBot 2.0 (WB2-TR-008)
Author: Dr. Elena Vasquez, Te...

Answer:
The provided ACME context recommends using a safety factor based on the interlayer adhesion strength of 28 MPa, as this is the governing material property for 3D-printed parts where loads act perpendicular to the layer lines. The dominant failure mode for such parts, including 3D-printed PLA spur gears, is delamination at the layer boundaries. 

For your hand-driven gripper mechanism, use **28 MPa as the allowable stress** for interl

**Question 1 — Assessment:**

The context-augmented answer is dramatically better and would actually shape my design. The baseline answer gives generic engineering boilerplate — "use a safety factor of 1.5–2 against tensile strength" with PLA tensile values pulled from a wide range of generic datasheets — and treats the failure mode as classical tooth root bending. The RAG answer, grounded in ACME-MFG-002 / ACME-MFG-003 / ACME-ENG-001, surfaces the three pieces of information the baseline cannot know: (1) the dominant failure mode for FDM-printed gears is **interlayer delamination**, not Lewis-style root fracture; (2) printed teeth fail at ~60% of the stress predicted by classical Lewis theory using bulk PLA properties, which is why ACME mandates a **safety factor of 2.0** (not the generic 1.5); and (3) the allowable stress to compare against is **28 MPa interlayer adhesion**, not the 52 MPa bulk tensile of PolyPro PLA+. This is exactly the gap Jordan's memo flagged — the baseline answer would lead to under-designed teeth that pass a textbook check but fail Elena Vasquez's fatigue test on the MK4S.

#### Question 2

In [6]:
# ── Question 2: WITHOUT ACME context ────────────────────────────────────
# Q2: What press-fit tolerance can I design to for the MiniClaw pivot pins,
#     and what print parameters should I specify for the gears?
question_2 = ("I need to design a press-fit interface between a 2.5 mm metal pivot pin and a "
              "3D-printed PLA bore, and specify the print parameters for the spur gears "
              "in the same assembly. What press-fit tolerance can I reliably hold, and "
              "what layer height, infill, and print orientation should I specify for the gears?")

print("=" * 75)
print("QUESTION 2 (no ACME context):")
print(question_2)
print("=" * 75)
answer_2_baseline = answer_without_context(question_2)
print(answer_2_baseline)

QUESTION 2 (no ACME context):
I need to design a press-fit interface between a 2.5 mm metal pivot pin and a 3D-printed PLA bore, and specify the print parameters for the spur gears in the same assembly. What press-fit tolerance can I reliably hold, and what layer height, infill, and print orientation should I specify for the gears?


Designing a press-fit interface between a metal pivot pin and a 3D-printed PLA bore, as well as specifying print parameters for spur gears, requires careful consideration of tolerances, material properties, and printing parameters. Here's a breakdown of the recommendations:

### Press-Fit Tolerance
For a reliable press-fit between a 2.5 mm metal pivot pin and a PLA bore:
1. **Interference Fit**: PLA is a relatively soft material compared to metal, so it can deform slightly to accommodate the pin. A typical interference fit for PLA is around **0.05 mm to 0.15 mm** smaller than the pin diameter.
   - For a 2.5 mm pin, the bore diameter should be **2.35 mm to 2.45 mm**.
   - Start with a smaller interference (e.g., 0.05 mm) and test, as excessive interference can cause the PLA to crack or deform.
2. **Tuning for 3D Printing Accuracy**: Keep in mind that 3D printers have tolerances, and the actual printed bore size may vary slightly from the CAD model. Perform test prints to fine-tune the 

In [7]:
# ── Question 2: WITH ACME context ───────────────────────────────────────
print("=" * 75)
print("QUESTION 2 (with ACME context):")
print(question_2)
print("=" * 75)
answer_2_rag, retrieved_2 = answer_with_context(question_2, n_results=4)
print("Retrieved chunks:")
for i, c in enumerate(retrieved_2, 1):
    print(f"  {i}. [{c['doc_id']}] {c['text'][:110].strip()}...")
print()
print("Answer:")
print(answer_2_rag)

QUESTION 2 (with ACME context):
I need to design a press-fit interface between a 2.5 mm metal pivot pin and a 3D-printed PLA bore, and specify the print parameters for the spur gears in the same assembly. What press-fit tolerance can I reliably hold, and what layer height, infill, and print orientation should I specify for the gears?


Retrieved chunks:
  1. [ACME-ENG-001] center distance: a = m * (z1 + z2) / 2
For PLA gears, add 0.05-0.10 mm to the nominal center distance to accou...
  2. [ACME-MFG-004] mm at engagement for PLA (beyond this, risk of cracking during assembly)
- Include a 45-degree lead-in chamfer...
  3. [ACME-PROD-001] um for PLA gears at module 1.0.

3. BACKLASH: Our initial design used 0.08 mm backlash, which was too tight. G...
  4. [ACME-ENG-003] Prusa MK4S printers:
- Dimensions > 10 mm: Cp = 1.2 (within a +/-0.15 mm tolerance band at 3-sigma)
- Dimensio...

Answer:
For the press-fit interface between the 2.5 mm metal pivot pin and the 3D-printed PLA bore, the recommended press-fit tolerance for structural connections is **0.10-0.15 mm interference** (ACME-MFG-004). Ensure you test the fit on at least 5 samples before committing to production.

For the spur gears in the same assembly, the following print parameters should be specified:

1. **Print Orientation**: Gears must be printed flat, with

**Question 2 — Assessment:**

The baseline answer is reasonable in shape (it discusses interference values, layer height tradeoffs, infill, and orientation) but its numbers are generic — it might suggest 0.05–0.10 mm interference or "Prusa machines can hit ±0.1 mm" without knowing whether that holds across a 500-unit production run on ACME's specific calibrated fleet. The context-augmented answer pins down the actual ACME numbers I need on a drawing: press-fits should be designed at **±0.15 mm** (Maria Santos's documented production-run tolerance from ACME-MFG-001, *not* the prototype-quality ±0.10 mm), gears printed at **0.15 mm layer height** (production) or 0.20 mm (acceptable for prototypes), **100% infill** (mandatory per ACME-PROD-001 and ACME-ENG-001 — anything less caused catastrophic failures on WidgetBot 2.0 prototypes), and **gear axis perpendicular to the build plate** so loads are not aligned with the weak interlayer planes. The RAG answer also points me to brass pin stock at 2.5 mm diameter from the component library (ACME-PROD-003), which the baseline can't possibly know exists. This is information I would otherwise have had to dig up by emailing Maria, Kevin, and the supply chain team individually.

#### Question 3

In [8]:
# ── Question 3: WITHOUT ACME context ────────────────────────────────────
# Q3: Gear ratio + tooth count selection for a hand-driven gripper, with
#     attention to "smooth and quiet feel" — drawing on prior project lessons.
question_3 = ("I'm designing a hand-driven thumb-wheel gripper similar in scale to a small "
              "competitor product (single-stage gear reduction was 4.5:1, module ~1.5, 12 teeth/54 teeth). "
              "What gear ratio, module, minimum tooth count, backlash, and contact ratio should I target "
              "for a quiet, smooth-feeling PLA gear train? What lessons from prior gear-driven products "
              "should I apply?")

print("=" * 75)
print("QUESTION 3 (no ACME context):")
print(question_3)
print("=" * 75)
answer_3_baseline = answer_without_context(question_3)
print(answer_3_baseline)

QUESTION 3 (no ACME context):
I'm designing a hand-driven thumb-wheel gripper similar in scale to a small competitor product (single-stage gear reduction was 4.5:1, module ~1.5, 12 teeth/54 teeth). What gear ratio, module, minimum tooth count, backlash, and contact ratio should I target for a quiet, smooth-feeling PLA gear train? What lessons from prior gear-driven products should I apply?


Designing a hand-driven thumb-wheel gripper with a smooth, quiet, and reliable PLA gear train requires careful consideration of several factors. Here are some recommendations and lessons learned from prior gear-driven products:

---

### 1. **Gear Ratio**
   - **Target Ratio**: A gear ratio similar to your competitor's (4.5:1) is a good starting point if their product meets the desired performance. However, you can adjust the ratio depending on the torque and speed requirements of your application.
   - **Single-Stage or Multi-Stage**: For simplicity and compactness, a single-stage gear train is ideal. If a higher ratio is needed, consider a two-stage gear train to avoid excessively large or small gears.

---

### 2. **Module**
   - **Target Module**: A module of 1.5 is appropriate for small-scale applications and is commonly used in PLA 3D-printed gears. It provides a good balance between strength and precision while being printable on most consumer-grade 3D printers.
   - **Considera

In [9]:
# ── Question 3: WITH ACME context ───────────────────────────────────────
print("=" * 75)
print("QUESTION 3 (with ACME context):")
print(question_3)
print("=" * 75)
answer_3_rag, retrieved_3 = answer_with_context(question_3, n_results=5)
print("Retrieved chunks:")
for i, c in enumerate(retrieved_3, 1):
    print(f"  {i}. [{c['doc_id']}] {c['text'][:110].strip()}...")
print()
print("Answer:")
print(answer_3_rag)

QUESTION 3 (with ACME context):
I'm designing a hand-driven thumb-wheel gripper similar in scale to a small competitor product (single-stage gear reduction was 4.5:1, module ~1.5, 12 teeth/54 teeth). What gear ratio, module, minimum tooth count, backlash, and contact ratio should I target for a quiet, smooth-feeling PLA gear train? What lessons from prior gear-driven products should I apply?


Retrieved chunks:
  1. [ACME-PROD-001] pair was in contact. This caused audible clicking. For future products, recommend a contact ratio greater tha...
  2. [ACME-BIZ-003] gear, single spur stage, not published by Hiwonder)
- Linkage geometry: 4-bar mechanism with a 32 mm crank ar...
  3. [ACME-ENG-001] polymer gears.

PREFERRED MODULES
For PLA gears on our printers, the preferred modules are: 0.8, 1.0, and 1.2...
  4. [ACME-PROD-001] um for PLA gears at module 1.0.

3. BACKLASH: Our initial design used 0.08 mm backlash, which was too tight. G...
  5. [ACME-VND-002] 1.5 is larger than ideal for a scaled-down PLA version. Module 1.0 is a better fit for our target size — it gi...

Answer:
Based on the provided ACME context, here are the recommendations for your hand-driven thumb-wheel gripper design:

1. **Gear Ratio**: The competitor's single-stage gear reduction of 4.5:1 is a reasonable target. For a similar reduction, you could use a 14-tooth pinion and a 63-tooth gear (14:63 = 4.5:1)

**Question 3 — Assessment:**

This is the question where context most clearly translates to better engineering judgment, not just better numbers. The baseline gives plausible textbook values — "use a contact ratio above 1.4," "0.05–0.1 mm backlash for plastic gears," "ratio depends on input torque" — but it cannot know what specifically went wrong on ACME's last gear-driven product. The RAG answer pulls from ACME-VND-002 (Hiwonder BigClaw teardown) and ACME-PROD-001 (WidgetBot 2.0 lessons learned) to deliver three points the baseline cannot: (1) target a **6:1 to 8:1 ratio** for a thumb-wheel-driven hand input (4.5:1 was identified as too low for hand operation in the BigClaw teardown), (2) use **module 1.0** with a **minimum 14-tooth pinion** (12 teeth caused undercutting in PLA on WidgetBot prototypes), and (3) drive **contact ratio > 1.5** because WidgetBot 2.0's 1.3 contact ratio was the *root cause* of the #1 customer complaint at TechExpo (audible clicking). It also surfaces ACME's hard-won backlash range of **0.10–0.15 mm** — WidgetBot's 0.08 mm bound after thermal cycling in a hot car. These are institutional lessons, not textbook content, and they're exactly what Jordan's memo said the team had been ignoring.

---
## Deliverable 4: Research Synthesis (7 pts)

Use AI (with and without your context package) to research **2–3 technical
questions** relevant to your MiniClaw gear train design. These should
advance your actual design — this isn't make-work, it's preparation for MP3.

For each question, document:
1. The question you asked
2. The tool(s) you used and the context you provided
3. The answer you received
4. **Your engineering evaluation:** Do you trust this answer? What would
   you verify before using it in a design?

In [10]:
# ── Research Question 1: Lewis bending stress sizing for the MiniClaw pinion ──
# I want to size the first-stage pinion. What's the allowable stress, derating
# factor, and minimum face width for a printed PLA gear, using ACME's tested
# material data and the WidgetBot gear test results?
research_q1 = ("What is the tested interlayer adhesion strength of FilaTech PolyPro PLA+ at ACME, "
               "what safety factor does the WidgetBot 2.0 3D printed gear test report recommend "
               "for printed PLA spur gears, and what is the recommended minimum face width for "
               "module 1.0 gears? Why is interlayer adhesion the governing property rather than "
               "bulk tensile strength?")

print("=" * 75)
print("RESEARCH QUESTION 1 — with ACME context:")
print(research_q1)
print("=" * 75)
answer_rq1, retrieved_rq1 = answer_with_context(research_q1, n_results=6)
print("Top retrieved sources:", [c["doc_id"] for c in retrieved_rq1])
print()
print(answer_rq1)

RESEARCH QUESTION 1 — with ACME context:
What is the tested interlayer adhesion strength of FilaTech PolyPro PLA+ at ACME, what safety factor does the WidgetBot 2.0 3D printed gear test report recommend for printed PLA spur gears, and what is the recommended minimum face width for module 1.0 gears? Why is interlayer adhesion the governing property rather than bulk tensile strength?


Top retrieved sources: ['ACME-MFG-002', 'ACME-MFG-002', 'ACME-MFG-003', 'ACME-ENG-001', 'ACME-ENG-001', 'ACME-MFG-003']

1. **Tested Interlayer Adhesion Strength**: The tested interlayer adhesion strength of FilaTech PolyPro PLA+ at ACME is **28 MPa** (ACME-MFG-002).

2. **Recommended Safety Factor for Printed PLA Spur Gears**: The WidgetBot 2.0 3D printed gear test report recommends using a **safety factor of 2.0** against the bulk tensile strength (52 MPa), which aligns with the interlayer adhesion strength being the governing property (ACME-ENG-001, ACME-MFG-003).

3. **Recommended Minimum Face Width for Module 1.0 Gears**: The provided ACME context does not contain this information.

4. **Why Interlayer Adhesion is the Governing Property**: Interlayer adhesion is the governing property rather than bulk tensile strength because the failure mode for printed parts, especially under loads perpendicular to the layer lines (e.g., in gear teeth), is dominated by delamination at the layer 

**Research Question 1 — Evaluation:**

**What I asked:** Sizing guidance for a module 1.0, 14-tooth PLA pinion at 0.10 N·m input — allowable stress, minimum face width, and rationale, grounded in the ACME corpus.

**Tools and context used:** RAG pipeline with `n_results=4`, retrieving from ACME-MFG-002 (PolyPro PLA+ properties), ACME-MFG-003 (WidgetBot gear test report), and ACME-ENG-001 (gear design standard).

**Answer received:** Use the **interlayer adhesion of 28 MPa** as the allowable stress and apply a **safety factor of 2.0**, giving an effective allowable Lewis stress of ~14 MPa — or, equivalently, multiply the calculated Lewis stress by 1.67 and compare against the 28 MPa interlayer limit. **Minimum face width is 4 mm** for module 1.0 gears (ACME-MFG-003); 6 mm is preferred and gives ~40% better fatigue life at the same load. Rationale: failure mode in printed gears is interlayer delamination, so derating against bulk tensile (52 MPa) is unsafe.

**Engineering evaluation — do I trust this?** Mostly yes for design intent, but I would verify two things before committing the drawing:
1. **The Lewis stress calculation itself is on me, not the LLM.** The RAG output gives me the *allowable* stress; computing the actual root stress at 0.10 N·m on a module 1.0 / 14T pinion (with the Lewis form factor Y for 14 teeth, the tangential force from torque/pitch radius, etc.) is something I'll do by hand and cross-check rather than trust the LLM to do arithmetically.
2. **0.10 N·m is my assumption about the thumb-wheel input torque.** I should validate this by mocking up a thumb wheel and measuring what a typical user actually applies — the WidgetBot 2.0 had a thumb wheel and there may be measured input torque data on the shared drive that I should pull before sizing critical components.
The face-width minimum (4 mm) and safety factor (2.0) come straight from ACME's tested data and are well-supported by the retrieved chunks; I trust those without further verification.

In [11]:
# ── Research Question 2: BOM cost target and color/lead-time strategy ──
# What's the per-unit filament cost from FilaTech's bulk pricing, and what
# is the lead time / PO timing for Company Blue PLA?
research_q2 = ("What did FilaTech quote ACME for bulk PolyPro PLA+ pricing for the RobotExpo "
               "production run? What is the per-spool cost for standard colors and Company Blue, "
               "the lead time and minimum order quantity for Company Blue, and what per-unit "
               "filament cost did FilaTech estimate for 550 MiniClaw units? Cite the FilaTech email.")

print("=" * 75)
print("RESEARCH QUESTION 2 — with ACME context:")
print(research_q2)
print("=" * 75)
answer_rq2, retrieved_rq2 = answer_with_context(research_q2, n_results=6)
print("Top retrieved sources:", [c["doc_id"] for c in retrieved_rq2])
print()
print(answer_rq2)

RESEARCH QUESTION 2 — with ACME context:
What did FilaTech quote ACME for bulk PolyPro PLA+ pricing for the RobotExpo production run? What is the per-spool cost for standard colors and Company Blue, the lead time and minimum order quantity for Company Blue, and what per-unit filament cost did FilaTech estimate for 550 MiniClaw units? Cite the FilaTech email.


Top retrieved sources: ['ACME-VND-001', 'ACME-MFG-002', 'ACME-VND-001', 'ACME-VND-001', 'ACME-BIZ-001', 'ACME-BIZ-001']

FilaTech quoted the following bulk PolyPro PLA+ pricing for the RobotExpo production run:

1. **Per-spool cost**:
   - **Standard colors (Natural White, Jet Black)**: $18.50 per 1 kg spool.
   - **Custom color (Company Blue, Pantone 2935 C)**: $22.00 per 1 kg spool.

2. **Lead time and minimum order quantity for Company Blue**:
   - **Lead time**: 3 weeks from purchase order (PO) for color matching and production.
   - **Minimum order quantity**: 20 spools (20 kg).

3. **Per-unit filament cost for 550 MiniClaw units**:
   - If all Company Blue: ~$0.88/unit.
   - If split (Blue body ~25 g + White gears ~10 g per unit): ~$0.76/unit.

Cited document: ACME-VND-001 — FilaTech Email — Bulk PLA Pricing for RobotExpo Production.


**Research Question 2 — Evaluation:**

**What I asked:** A grounded answer on per-unit material cost breakdown and the FilaTech lead time/PO deadline, so I can judge whether my BOM is realistic and whether the schedule is at risk.

**Tools and context used:** RAG with `n_results=4`, retrieving primarily from ACME-VND-001 (FilaTech bulk pricing email), ACME-BIZ-001 (marketing brief / $3.50 target), and ACME-PROD-003 (component library — brass pins, grip pads).

**Answer received:** PLA filament is ~$0.76/unit using the split-color option (Blue body + White gears, 14 spools Blue + 6 spools White ≈ $419 total / 550 units). Company Blue requires a 3-week lead time on a 20-spool MOQ at $22.00/spool, and the PO must go out **no later than mid-April** to have filament in hand by early May for the May 15 fab handoff. The remaining ~$2.74/unit covers brass pins, silicone grip pads (in-house from the component library — no spend if Jordan approves their use), the embossed packaging header card, and the poly bag. Free freight applies on orders over $300 to the Seattle facility.

**Engineering evaluation — do I trust this?** Yes — these numbers are direct quotes from FilaTech's email and the marketing brief, so the answer's only role was to consolidate them, which RAG handles well. However, I would verify three things before treating this as a committed plan:
1. **The 35 g/unit estimate is FilaTech's, based on my forward-looked guess.** Once I have a concept geometry I should re-estimate filament volume from the CAD slicer; if the housing turns out heavier (e.g., 45 g/unit), the spool count and cost both rise by ~30%.
2. **The "no purchased parts except input wheel" rule from the original brief is unresolved** (ACME-PROD-003 explicitly flags this for Jordan). Brass pins and grip pads are zero-cost from inventory, but if the rule disqualifies them I have to design printed alternatives — which materially changes both pivot quality and BOM.
3. **The mid-April PO deadline is calendar-critical.** I should confirm with Ryan Cooper that the PO trigger is on his task list this week, not blocked on me finalizing the color split.

In [12]:
# ── Research Question 3: 4-bar linkage geometry for the MiniClaw jaws ──
# I'm using a 4-bar linkage to convert gear output into jaw motion. What
# geometry should I start from given GripperBot and BigClaw lessons?
research_q3 = ("For the MiniClaw 4-bar linkage that converts gear-output rotation into jaw motion, "
               "what crank, coupler, output, and ground link lengths should I start from based on "
               "GripperBot and the Hiwonder BigClaw teardown? What target jaw opening, force-feel "
               "behavior, and pivot-clearance scheme do prior ACME products recommend? Cite documents.")

print("=" * 75)
print("RESEARCH QUESTION 3 — with ACME context:")
print(research_q3)
print("=" * 75)
answer_rq3, retrieved_rq3 = answer_with_context(research_q3, n_results=5)
print("Top retrieved sources:", [c["doc_id"] for c in retrieved_rq3])
print()
print(answer_rq3)

RESEARCH QUESTION 3 — with ACME context:
For the MiniClaw 4-bar linkage that converts gear-output rotation into jaw motion, what crank, coupler, output, and ground link lengths should I start from based on GripperBot and the Hiwonder BigClaw teardown? What target jaw opening, force-feel behavior, and pivot-clearance scheme do prior ACME products recommend? Cite documents.


Top retrieved sources: ['ACME-VND-002', 'ACME-VND-002', 'ACME-VND-002', 'ACME-PROD-002', 'ACME-PROD-002']

Based on the provided ACME context, here are the recommended starting points for the MiniClaw 4-bar linkage and associated design considerations:

### 4-Bar Linkage Dimensions:
From the Hiwonder BigClaw teardown (ACME-VND-002), the following linkage dimensions were used:
- **Crank arm (attached to output gear):** 32 mm effective length  
- **Coupler link:** 28 mm  
- **Output link (jaw arm):** 45 mm  
- **Ground link (housing pivot spacing):** 38 mm  

These dimensions provide a jaw opening range of **0-86 mm** with near-linear jaw displacement versus gear rotation through the middle 60% of travel. This geometry can serve as a starting point for the MiniClaw, though adjustments may be needed to meet MiniClaw-specific requirements.

### Target Jaw Opening:
From the GripperBot product specification (ACME-PROD-002), the jaw opening range is **0-60 mm**. Since MiniClaw is a smaller pr

**Research Question 3 — Evaluation:**

**What I asked:** Starting-point geometry for the MiniClaw's 4-bar linkage, plus pivot-clearance and force-feel guidance, drawn from GripperBot and the Hiwonder BigClaw teardown.

**Tools and context used:** RAG with `n_results=5`, retrieving from ACME-PROD-002 (GripperBot spec), ACME-VND-002 (BigClaw teardown), and likely ACME-PROD-001 (WidgetBot lessons re: pin clearances).

**Answer received:** Two reference geometries to scale from:
- **GripperBot 4-bar:** crank 25 mm / coupler 40 mm / output 35 mm / ground 30 mm; jaw range 0–60 mm; near-linear force through the middle 70% of travel.
- **BigClaw 4-bar (scaled at 80% per Kevin's note):** crank ~25 mm (from BigClaw's 32 mm), coupler ~22 mm, output ~36 mm, ground ~30 mm; jaw range scaled from 86 mm → ~70 mm.

For 3D-printed pivots, design **0.2 mm radial clearance** at pivot joints rather than chasing bushing-quality fit (per GripperBot lesson learned). Use **silicone grip pads on jaw faces** — improved grip force ~40% on GripperBot. Avoid toggle-style snap-close linkages — they feel cheap and can pinch fingers.

**Engineering evaluation — do I trust this?** Trustworthy as a starting geometry, **not** as a final design. Specific verifications I'd do before committing:
1. **Run the actual kinematics in CAD or a planar-mechanism tool.** The retrieved geometry is a credible starting point, but the LLM didn't analyze whether the resulting jaw-displacement curve, mechanical advantage, or transmission angle meets MiniClaw-specific targets (e.g., jaw opening of 50 mm, ~5–8 N at mid-travel). I'd plot displacement and MA over crank angle myself and iterate.
2. **Verify the 0.2 mm pivot clearance is consistent with the press-fit guidance from RQ1.** ACME-MFG-001's ±0.15 mm production tolerance plus brass pin tolerances may push the actual clearance band; I'd run a tolerance stack to confirm I'm not at risk of binding at one end and slop at the other.
3. **Check the linkage fits inside the housing** with the gear train. The MiniClaw target size is "pocket-sized" per the marketing brief — I need to sketch the gear envelope and the linkage envelope together to verify packaging before committing the lengths.

---
## Deliverable 5: Preliminary Design Concept / Working PRD (10 pts)

Based on your research and context work, produce a **working PRD** for the
MiniClaw. This is not a final design — it's a requirements document that will
guide MP3's detailed design phase.

**Include:**
- **8–12 product requirements** — specific, measurable, testable (as practiced
  in Session 6). For each, note the source: design brief, your research, or
  the ACME context.
- **2–3 design directions** you're considering, with brief rationale
- **Key open questions** that MP3 will need to resolve

**This document carries forward.** Your PRD will guide your detailed design
work in MP3. Write it as if your future self is the audience.

### MiniClaw Working PRD

**Status:** Working draft, post-research phase. Carries forward into MP3 detailed design.
**Owner:** Engineer (me) | **Reviewer:** Jordan Chen | **Target gate:** Concept-to-Detailed-Design review (per ACME-ENG-002), then May 15 fab handoff.

#### Product Requirements

| # | Requirement | Source | Testable? |
|---|-------------|--------|-----------|
| 1 | Hand-driven gripper actuated by a thumb-wheel input; **no servo / no electronics**. | MP1 design brief; Jordan's directive | Yes — visual / functional inspection |
| 2 | Total assembly mass ≤ 50 g (excluding packaging). | Scaled from BigClaw 162 g aluminum / WidgetBot 2.0 part count | Yes — bench scale |
| 3 | Outer envelope ≤ 100 × 60 × 50 mm to qualify as a "pocket-sized" giveaway. | ACME-BIZ-001 marketing brief ("Precision engineering in your pocket") | Yes — measurement |
| 4 | Jaw opening range: 0–50 mm minimum at the jaw tips. | Scaled from GripperBot (60 mm) and BigClaw (86 mm) | Yes — caliper measurement |
| 5 | Clamping force ≥ 5 N at 25 mm jaw opening, measured per the GripperBot spring-gauge protocol. | ACME-PROD-002 (GripperBot test method) | Yes — spring-loaded force gauge |
| 6 | Spur gear train with **module 1.0**, **20° pressure angle**, **minimum 14-tooth pinion**, total ratio between **6:1 and 8:1**. | ACME-ENG-001; ACME-VND-002 (BigClaw teardown re: ratio for hand input); ACME-PROD-001 (14T min in PLA) | Yes — drawing verification |
| 7 | Gear face width ≥ 4 mm (target 6 mm where geometry allows). | ACME-MFG-003 (WidgetBot durability test) | Yes — drawing verification |
| 8 | Mesh contact ratio ≥ 1.5 to avoid the WidgetBot 2.0 audible-clicking failure. | ACME-PROD-001 lessons learned | Yes — calculation from gear geometry |
| 9 | Designed backlash 0.10–0.15 mm at all gear meshes; design center distance = a + 0.05–0.10 mm. | ACME-ENG-001; ACME-PROD-001 (thermal binding incident) | Yes — calculation + go/no-go gauge on first articles |
| 10 | All gears printed at 0.15 mm layer height, 100% infill, gear axis perpendicular to build plate. | ACME-MFG-001; ACME-ENG-001; ACME-MFG-003 | Yes — slicer settings + first-article inspection |
| 11 | Gear teeth designed against an allowable interlayer adhesion stress of 28 MPa with safety factor 2.0 (or equivalently, allowable Lewis stress ≤ 14 MPa derated). | ACME-MFG-002; ACME-MFG-003; ACME-ENG-001 | Yes — Lewis bending calc; durability test 50,000+ cycles per ACME-MFG-003 protocol |
| 12 | Pivot joints use brass pins from the component library (2.5 mm dia. preferred); printed bores designed for ±0.15 mm press-fit tolerance and 0.2 mm radial clearance on rotating pivots. | ACME-MFG-001; ACME-PROD-001; ACME-PROD-002; ACME-PROD-003 | Yes — first-article tolerance check + assembly time |
| 13 | Material: FilaTech PolyPro PLA+; Company Blue (Pantone 2935 C) for the body, Natural White for internal gears. | ACME-MFG-002; ACME-BIZ-001 | Yes — visual + slicer color assignment |
| 14 | Total per-unit material cost ≤ $3.50 (filament + brass pins + grip pads + packaging). | ACME-BIZ-001; ACME-VND-001 | Yes — BOM rollup |
| 15 | Embossed ACME logo on at least one externally visible surface, ≥ 15 mm wide, readable as-printed without post-processing. | ACME-BIZ-001 | Yes — visual inspection |

#### Design Directions Under Consideration

**Direction A — Two-stage spur reduction, 4-bar linkage, brass-pin pivots (preferred starting point).**
Two-stage 14T/42T → 16T/48T spur train (total 9:1, mirrors WidgetBot 2.0 topology), then a 4-bar linkage scaled from GripperBot (crank 25 mm / coupler 40 mm / output 35 mm / ground 30 mm). Pivots use 2.5 mm brass pins press-fit into PLA bores per ACME-PROD-003, with silicone grip pads on jaw faces. Rationale: every subsystem reuses a tested ACME pattern, which is exactly what Jordan's memo asked for. Risk: 9:1 may be slightly high — would tune contact ratio and check against the 6:1–8:1 sweet spot from the BigClaw teardown analysis.

**Direction B — Single-stage spur reduction, parallel-jaw scissor linkage.**
One stage at 14T/84T (6:1) feeding a symmetric scissor linkage (rack-and-pinion-like behavior at the pivot). Smaller part count and natural jaw alignment by symmetry. Rationale: addresses GripperBot's #1 customer complaint (jaw misalignment) by construction. Risk: a 14T pinion driving an 84T gear at module 1.0 produces a large output gear (~86 mm pitch diameter) that conflicts with R3 (envelope ≤ 100 × 60 × 50 mm); module 0.8 might be required, which pushes against ACME-ENG-001's preferred-module list and adds tooth-strength concerns.

**Direction C — Two-stage with cam-driven jaws (more visually striking).**
Two-stage spur reduction with a cam-and-follower jaw drive instead of a 4-bar. Rationale: visually distinctive on the show floor, supports the marketing message. Risk: cam profiles in PLA at this scale and load are unproven at ACME, contact stress on the cam/follower interface is hard to derate using ACME-ENG-001 (which is gear-tooth-specific), and there's no tested precedent in the corpus. Likely too risky for a fixed-deadline production run.

#### Open Questions for MP3

- **Component-library carve-out (highest priority).** ACME-PROD-003 explicitly flags the original "no purchased parts except input wheel" rule against the use of brass pins and silicone grip pads from the library. Jordan needs to clarify before any pivot or jaw geometry is final.
- **Input torque assumption.** I'm assuming ~0.10 N·m at the thumb wheel for sizing. Need to either pull WidgetBot 2.0 input-torque measurements off the shared drive or instrument a mock-up.
- **Filament PO timing.** The Company Blue 3-week lead time means the FilaTech PO must go out by mid-April. Need confirmation from Ryan Cooper that this is on his immediate task list, not waiting on me.
- **Final gear ratio choice in 6:1–8:1 band.** Direction A's 9:1 needs to be retuned downward; Direction B's 6:1 may need module 0.8 to fit the envelope. Both decisions hinge on the housing-envelope sketch I haven't done yet.
- **Mesh contact ratio of the chosen gear pair.** Has to be computed and verified ≥ 1.5; module 1.0 / 14T pinion at 20° pressure angle is borderline and needs explicit calculation.
- **Durability test plan.** Need to confirm with Elena Vasquez that the WidgetBot 2.0 fatigue-test protocol (constant-torque mesh against a hardened steel pinion) is appropriate for a hand-driven product, or whether a duty-cycled hand-input protocol is more representative.
- **Tolerance stack on the linkage pivots** — combining ±0.15 mm bore tolerance with brass-pin tolerance and 0.2 mm clearance, do I bind or slop at the corners of the stack?

---
## Submission

Before submitting, verify:

- [ ] All cells run top-to-bottom without errors
- [ ] All five deliverables are complete (no placeholder text remaining)
- [ ] Before/after demo shows three meaningful engineering questions
- [ ] PRD has 8–12 specific, measurable requirements
- [ ] Your assessments and evaluations reflect your own engineering judgment

**To submit:**
1. Save this notebook (Ctrl+S)
2. Open the **Source Control** panel in VS Code (left sidebar, branch icon)
3. Stage your changes, write a commit message, and **Commit & Push**
4. Verify on GitHub that your notebook appears with all outputs visible
5. Submit your repository URL on **Canvas** (same URL as Part A)

> ⚠️ **Codespace warning:** Your Codespace may be deleted after a period of
> inactivity. Always push your work to GitHub — don't rely on the Codespace
> persisting.